In [1]:
import numpy as np
import pymc as pm
import arviz as az
import os
import re
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
from bs4 import BeautifulSoup
import pickle
from datetime import time, datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import numpy as np
from scipy.optimize import minimize

import numpy as np
from scipy.stats import norm
import random

random.seed(42)
import warnings
warnings.filterwarnings("ignore")

In [2]:
BAD_PIDS=['robas+120@developers.pg.com',
 'robas+171@developers.pg.com',
 'robas+146@developers.pg.com',
 'robas+157@developers.pg.com',
 'robas+170@developers.pg.com',
 'robas+161@developers.pg.com',
 'robas+180@developers.pg.com',
 'robas+173@developers.pg.com','robas+152@developers.pg.com']

In [3]:
script_dir = os.getcwd()
ROOT_PATH = os.path.join(script_dir, '../../OralyticsMRT/')

ORALYTICS_MRT_FILE = os.path.join(ROOT_PATH, "oralytics_mrt_data.csv")
MRT_PARTICIPANTS_IDS_FILE = os.path.join(ROOT_PATH, "mrt_participant_ids.csv")
RL_PATH_PREFIX = os.path.join(ROOT_PATH, "rl data tables/")
ACTION_SELECTION_TABLE = os.path.join(RL_PATH_PREFIX, "action_selection_data_table.csv")
USER_INFO_FILE = os.path.join(RL_PATH_PREFIX, "user_info_table.csv")
RAW_BRUSHING_INFO_FILE = os.path.join(ROOT_PATH, 'raw_brushing_info.html')
EXTRACTED_SEGMENTS_FILE = os.path.join(os.getcwd(), 'data_tables', 'extracted_device_segments.txt')

# Load datasets
anna_df = pd.read_csv(ORALYTICS_MRT_FILE)
anna_df = anna_df.sort_values(by=['user_id', 'decision_time'])
anna_df = anna_df[~anna_df['user_id'].isin(BAD_PIDS)]

mrt_df = pd.read_csv(MRT_PARTICIPANTS_IDS_FILE)
mrt_df = mrt_df[~mrt_df['LY Email'].isin(BAD_PIDS)]
mrt_id_mapping = dict(zip(mrt_df['LY ID'], mrt_df['LY Email']))
reverse_mrt_id_mapping = dict(zip(mrt_df['LY Email'], mrt_df['LY ID']))
MRT_LY_ID = list(reverse_mrt_id_mapping.keys())
# Load action selection table
ACTION_SELECTION = pd.read_csv(ACTION_SELECTION_TABLE)

# Load user information
USER_INFO_DF = pd.read_csv(USER_INFO_FILE)

# T0_PATH_PREFIX = os.path.join(ROOT_PATH, "Oralytics_simulation/data_tables/")
# T0_TABLE = os.path.join(T0_PATH_PREFIX, "user_t0_ranges.csv")
# T0_RANGE_DF = pd.read_csv(T0_TABLE)

In [4]:


USER_BRUSHING_PATH = os.path.join(T0_PATH_PREFIX, "user_brushing_info.pkl")
with open(USER_BRUSHING_PATH, 'rb') as file:
    all_user_brushing = pickle.load(file)


all_user_app_opening={}
for user_id in all_user_brushing.keys():
    USER_APP_OPENING_PATH = os.path.join(T0_PATH_PREFIX, "app_openingtime/")
    all_user_app_opening[user_id] = np.load(USER_APP_OPENING_PATH+user_id+'.npy',allow_pickle=True)

In [10]:
import os
import json
from datetime import datetime, date
from bisect import bisect_right
from datetime import timedelta
# Date range
start_date = date(2023, 12, 1)
end_date = date(2024, 2, 9)

def get_user_info(col_name, user_id):
    return USER_INFO_DF[USER_INFO_DF['user_id'] == user_id][col_name].values[0]

for user_id in ['digitaldentalcoach+234@gmail.com']:
    start_user_date = datetime.strptime(get_user_info("user_start_day", user_id), '%Y-%m-%d').date()
    end_user_date = datetime.strptime(get_user_info("user_end_day", user_id), '%Y-%m-%d').date()

    morning_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['morning_time_weekday'].values[0]
    morning_decision_time_weekday = datetime.strptime(morning_decision_time, '%H:%M:%S')
    morning_decision_time_weekday=morning_decision_time_weekday.time()

    morning_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['morning_time_weekend'].values[0]
    morning_decision_time_weekend = datetime.strptime(morning_decision_time, '%H:%M:%S')
    morning_decision_time_weekend=morning_decision_time_weekend.time()

    
    evening_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['evening_time_weekday'].values[0]
    evening_decision_time_weekday = datetime.strptime(evening_decision_time, '%H:%M:%S')
    evening_decision_time_weekday=evening_decision_time_weekday.time()

    evening_decision_time=USER_INFO_DF[USER_INFO_DF['user_id'] == user_id]['evening_time_weekend'].values[0]
    evening_decision_time_weekend = datetime.strptime(evening_decision_time, '%H:%M:%S')
    evening_decision_time_weekend=evening_decision_time_weekend.time()

    APP_DATA_PATH = os.path.join(ROOT_PATH, "main controller data", "app analytics data", f"app_data_{user_id}.json")
    try:
        with open(APP_DATA_PATH, 'r') as file:
            app_data = json.load(file)
    except FileNotFoundError:
        print(f"No app data file found for user {user_id}")
        continue



    records = []

    current_date = start_user_date
    while current_date <= end_user_date:
        is_weekend = current_date.weekday() >= 5  # 5 = Saturday, 6 = Sunday

        # Morning
        morning_time = morning_decision_time_weekend if is_weekend else morning_decision_time_weekday
        morning_dt = datetime.combine(current_date, morning_time)
        records.append({
            "date": current_date,
            "period": "morning",
            "decision_time": morning_dt,
            "action": 0,
            "message_type": None
        })

        # Evening
        evening_time = evening_decision_time_weekend if is_weekend else evening_decision_time_weekday
        evening_dt = datetime.combine(current_date, evening_time)
        records.append({
            "date": current_date,
            "period": "evening",
            "decision_time": evening_dt,
            "action": 0,
            "message_type": None
        })

        current_date += timedelta(days=1)
    df = pd.DataFrame(records)
    

    all_filtered = []
    app_scheduling_times=[]


    message_schedule_dict={}

    for record in app_data:
        if 'analytics_data' in record:
            try:
                analytics = json.loads(record['analytics_data'])
                success= False
                for entry in analytics:
                    event = entry.get("event", "")
                    if( 'MainControllerDataRequested' in event):
                        success= True
                        break
                if(not success):
                    continue
                for entry in analytics:
                    event = entry.get("event", "")

                    if event.startswith("ScheduledMessage:"):
                        try:
                            event_time=datetime.strptime(event.split(",")[1], "%Y-%m-%d %H:%M:%S")
                            event_time = event_time.replace(second=0, microsecond=0)
                            if start_user_date <= event_time.date() <= end_user_date:
                                app_time=entry.get("app_created_at","")
                                app_time=datetime.strptime(app_time, "%Y-%m-%d %H:%M:%S")
                                app_time = app_time.replace(second=0, microsecond=0)
                                if(app_time not in message_schedule_dict):
                                    message_schedule_dict[app_time]=[]
                                if(app_time <= event_time):
                                    message_schedule_dict[app_time].append([event_time,event.split(",")[0].split(":")[1]])
                        except (IndexError, ValueError):
                            continue
            except json.JSONDecodeError:
                print(f"Failed to parse analytics_data for email {record.get('email')}")

    sorted_message_times = sorted(message_schedule_dict.keys())

    
    updated_records = []
    #print("\n")
    for idx, row in df.iterrows():
        decision_time = row['decision_time']


        matched_time = None
        min_diff = timedelta.max  # Start with maximum possible difference

        for t in sorted_message_times:
            if t <= decision_time:
                diff = decision_time - t
                if diff < min_diff:
                    min_diff = diff
                    matched_time = t
            else:
                break  # Since the list is sorted, stop early


        message_type = None
        action = 0
        if matched_time:
            for msg_time, msg_type in message_schedule_dict[matched_time]:
                if (msg_time==row['decision_time']):
                    message_type=msg_type
                    action=1
                    break
        updated_records.append({
            **row,
            "matched_app_time": matched_time,
            "action": action,
            "message_type": message_type
        })

    df_updated = pd.DataFrame(updated_records)

    print(len(df_updated[df_updated['action']==1]),user_id)
    
    # Save df_updated as CSV with user_id as filename
    csv_filename = f"{user_id}.csv"
    df_updated.to_csv(csv_filename, index=False)
    print(f"Saved DataFrame to {csv_filename}")
    
    

57 digitaldentalcoach+234@gmail.com
Saved DataFrame to digitaldentalcoach+234@gmail.com.csv


df_updated[df_updated['action']==1]

df_updated[df_updated['action']==1]

df_updated[df_updated['action']==1]

df_updated[df_updated['action']==1]

df_updated[df_updated['action']==1]

df_updated[df_updated['action']==1]

In [41]:
sorted_message_times

[datetime.datetime(2023, 10, 19, 7, 45),
 datetime.datetime(2023, 10, 19, 7, 46),
 datetime.datetime(2023, 10, 19, 17, 14),
 datetime.datetime(2023, 10, 20, 0, 51),
 datetime.datetime(2023, 10, 20, 10, 55),
 datetime.datetime(2023, 10, 20, 10, 56),
 datetime.datetime(2023, 10, 20, 11, 28),
 datetime.datetime(2023, 10, 20, 21, 44),
 datetime.datetime(2023, 10, 20, 22, 56),
 datetime.datetime(2023, 10, 20, 23, 15),
 datetime.datetime(2023, 10, 20, 23, 20),
 datetime.datetime(2023, 10, 20, 23, 23),
 datetime.datetime(2023, 10, 21, 14, 7),
 datetime.datetime(2023, 10, 22, 17, 49),
 datetime.datetime(2023, 10, 23, 0, 44),
 datetime.datetime(2023, 10, 23, 10, 49),
 datetime.datetime(2023, 10, 25, 10, 54),
 datetime.datetime(2023, 10, 25, 11, 10),
 datetime.datetime(2023, 10, 25, 13, 52),
 datetime.datetime(2023, 10, 25, 14, 9),
 datetime.datetime(2023, 10, 25, 14, 23),
 datetime.datetime(2023, 10, 25, 14, 51),
 datetime.datetime(2023, 10, 26, 2, 22),
 datetime.datetime(2023, 10, 26, 11, 35),

In [37]:
sorted_message_times

[datetime.datetime(2023, 10, 19, 7, 45),
 datetime.datetime(2023, 10, 19, 7, 46),
 datetime.datetime(2023, 10, 19, 17, 14),
 datetime.datetime(2023, 10, 20, 0, 51),
 datetime.datetime(2023, 10, 20, 10, 55),
 datetime.datetime(2023, 10, 20, 10, 56),
 datetime.datetime(2023, 10, 20, 11, 28),
 datetime.datetime(2023, 10, 20, 21, 44),
 datetime.datetime(2023, 10, 20, 22, 56),
 datetime.datetime(2023, 10, 20, 23, 15),
 datetime.datetime(2023, 10, 20, 23, 20),
 datetime.datetime(2023, 10, 20, 23, 23),
 datetime.datetime(2023, 10, 21, 14, 7),
 datetime.datetime(2023, 10, 22, 17, 49),
 datetime.datetime(2023, 10, 23, 0, 44),
 datetime.datetime(2023, 10, 23, 10, 49),
 datetime.datetime(2023, 10, 25, 10, 54),
 datetime.datetime(2023, 10, 25, 11, 10),
 datetime.datetime(2023, 10, 25, 13, 52),
 datetime.datetime(2023, 10, 25, 14, 9),
 datetime.datetime(2023, 10, 25, 14, 23),
 datetime.datetime(2023, 10, 25, 14, 51),
 datetime.datetime(2023, 10, 26, 2, 22),
 datetime.datetime(2023, 10, 26, 11, 35),

In [51]:
df_updated[df_updated['action']==1]

,date,period,decision_time,action,message_type,matched_app_time
1,2023-10-20,evening,2023-10-20 21:30:00,1,FB-09,2023-10-19 17:14:00
2,2023-10-21,morning,2023-10-21 08:00:00,1,QA-20,2023-10-19 17:14:00
5,2023-10-22,evening,2023-10-22 21:30:00,1,RP-20,2023-10-19 17:14:00
6,2023-10-23,morning,2023-10-23 06:30:00,1,QA-23,2023-10-23 00:44:00
10,2023-10-25,morning,2023-10-25 06:30:00,1,QA-12,2023-10-23 00:44:00
12,2023-10-26,morning,2023-10-26 06:30:00,1,QA-06,2023-10-26 02:22:00
13,2023-10-26,evening,2023-10-26 21:30:00,1,FB-13,2023-10-26 21:30:00
15,2023-10-27,evening,2023-10-27 21:30:00,1,FB-08,2023-10-26 21:30:00
23,2023-10-31,evening,2023-10-31 21:30:00,1,SR-42,2023-10-31 01:28:00
24,2023-11-01,morning,2023-11-01 06:30:00,1,QA-19,2023-10-31 01:28:00


In [29]:
pd.set_option('display.max_rows', None)

# Optional: Show all columns too, if needed
pd.set_option('display.max_columns', None)

# Optional: Prevent line wrapping in columns
pd.set_option('display.max_colwidth', None)

# Now display the DataFrame
print(df_updated)

           date   period       decision_time  action message_type  \
0    2023-11-05  morning 2023-11-05 08:00:00       0         None   
1    2023-11-05  evening 2023-11-05 22:00:00       0         None   
2    2023-11-06  morning 2023-11-06 06:30:00       0         None   
3    2023-11-06  evening 2023-11-06 22:00:00       0         None   
4    2023-11-07  morning 2023-11-07 06:30:00       1        RP-45   
5    2023-11-07  evening 2023-11-07 22:00:00       0         None   
6    2023-11-08  morning 2023-11-08 06:30:00       0         None   
7    2023-11-08  evening 2023-11-08 22:00:00       1        RP-24   
8    2023-11-09  morning 2023-11-09 06:30:00       0         None   
9    2023-11-09  evening 2023-11-09 22:00:00       1        SR-18   
10   2023-11-10  morning 2023-11-10 06:30:00       1        RP-18   
11   2023-11-10  evening 2023-11-10 22:00:00       0         None   
12   2023-11-11  morning 2023-11-11 08:00:00       0         None   
13   2023-11-11  evening 2023-11-1

In [15]:
app_data

[{'email': 'robas+143@developers.pg.com',
  'analytics_data': '[{"event": "GetScheduledMessagesRequest", "app_created_at": "2023-12-11 15:35:53", "id": "59039422-693c-4707-ba6d-73f0346f83e6"}, {"event": "GetScheduledMessagesError", "app_created_at": "2023-12-11 15:35:54", "id": "b152a49d-7e63-432f-91f3-8b44e7a78b32"}]',
  'app_created_at': '2023-12-11 15:35:55',
  'created_at': '2023-12-11 23:35:55'},
 {'email': 'robas+143@developers.pg.com',
  'analytics_data': '[]',
  'app_created_at': '2023-12-11 15:36:25',
  'created_at': '2023-12-11 23:36:25'},
 {'email': 'robas+143@developers.pg.com',
  'analytics_data': '[]',
  'app_created_at': '2023-12-11 15:36:55',
  'created_at': '2023-12-11 23:36:55'},
 {'email': 'robas+143@developers.pg.com',
  'analytics_data': '[]',
  'app_created_at': '2023-12-11 15:37:25',
  'created_at': '2023-12-11 23:37:25'},
 {'email': 'robas+143@developers.pg.com',
  'analytics_data': '[]',
  'app_created_at': '2023-12-11 15:37:55',
  'created_at': '2023-12-11 23: